# Phishing Website Detection — 02: Data Acquisition

Standalone. Run this after `01_setup_and_data.ipynb` has created the folder
structure, or on its own — Cell 1 rebuilds anything missing.

Every download is **idempotent** (cached on Drive) and **verified** (SHA-256 +
expected row count). A `PROVENANCE.json` is written to `data/raw/`, and a
`DATASETS.md` table is generated for the Data section of the manuscript.

### Registry

| Key | Source | Expected rows | Label column | Phishing encoded as |
|---|---|---|---|---|
| `phiusiil` | UCI id=967 | 235,795 | `label` | `0` |
| `uci2012` | UCI id=327 | 11,055 | `Result` | `-1` |
| `uci_multiclass` | UCI id=379 | 1,353 | `Result` | `-1` (3-class) |
| `hannousse` | Mendeley `c2gw7fy2j4` v3 | 11,430 | `status` | `"phishing"` |
| `vrbancic` | Mendeley `72ptz43s9v` v1 | 88,647 | `phishing` | `1` |
| `tan` | Mendeley `h3cgnj8hft` v1 | 10,000 | `CLASS_LABEL` | `1` |
| `tranco` | tranco-list.eu | 1,000,000 | — | benign reference |

`uci_multiclass` carries a *suspicious* third class — useful if you want to argue
that binary framing discards information, but drop it for the main experiments.

## Cell 1 — Environment (safe to re-run)

In [1]:
!pip install -q ucimlrepo pandas

from google.colab import drive
from pathlib import Path
import sys, os

drive.mount('/content/drive')

PROJECT   = "phishing-detection"
REPO_DIR  = Path("/content") / PROJECT
DRIVE_DIR = Path("/content/drive/MyDrive/research") / PROJECT

for d in ["data/raw", "data/external", "data/interim", "data/processed",
          "checkpoints", "models", "logs", "notebooks"]:
    (DRIVE_DIR / d).mkdir(parents=True, exist_ok=True)

for d in ["src", "src/data", "src/utils", "config", "reports", "reports/tables"]:
    (REPO_DIR / d).mkdir(parents=True, exist_ok=True)
for d in ["src", "src/data", "src/utils"]:
    (REPO_DIR / d / "__init__.py").touch()

for name in ["data", "checkpoints", "models", "logs"]:
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        raise RuntimeError(f"{link} exists and is not a symlink")
    link.symlink_to(DRIVE_DIR / name)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

RAW = REPO_DIR / "data" / "raw"
EXT = REPO_DIR / "data" / "external"
print("raw ->", RAW.resolve())
print("ext ->", EXT.resolve())

Mounted at /content/drive
raw -> /content/drive/MyDrive/research/phishing-detection/data/raw
ext -> /content/drive/MyDrive/research/phishing-detection/data/external


## Cell 1b — Checkpoint module (self-contained)

`/content` is wiped on every runtime reset, so this notebook cannot rely on
`checkpoint.py` having been written by notebook 01. It restores the module from
the Drive backup if present, and writes it from embedded source otherwise.

In [2]:
CKPT_SRC = 'import json, time, hashlib, subprocess\nfrom pathlib import Path\nimport joblib\n\n\ndef _git_hash(repo="/content/phishing-detection"):\n    try:\n        return subprocess.check_output(\n            ["git", "-C", repo, "rev-parse", "--short", "HEAD"],\n            stderr=subprocess.DEVNULL).decode().strip()\n    except Exception:\n        return "nogit"\n\n\nclass Checkpoint:\n    # Resumable checkpointing for Colab. Survives runtime resets via Drive.\n\n    def __init__(self, root="/content/phishing-detection/checkpoints",\n                 run_id=None, verbose=True):\n        self.root = Path(root)\n        self.run_id = run_id or time.strftime("run_%Y%m%d_%H%M%S")\n        self.dir = self.root / self.run_id\n        self.dir.mkdir(parents=True, exist_ok=True)\n        self.verbose = verbose\n        self.manifest_path = self.dir / "manifest.json"\n        self.manifest = (json.loads(self.manifest_path.read_text())\n                         if self.manifest_path.exists() else {})\n\n    @staticmethod\n    def _key(params):\n        if params is None:\n            return "default"\n        blob = json.dumps(params, sort_keys=True, default=str)\n        return hashlib.md5(blob.encode()).hexdigest()[:10]\n\n    def _path(self, stage, params=None):\n        return self.dir / (stage + "__" + self._key(params) + ".joblib")\n\n    def exists(self, stage, params=None):\n        return self._path(stage, params).exists()\n\n    def save(self, stage, obj, params=None, meta=None):\n        p = self._path(stage, params)\n        joblib.dump(obj, p, compress=3)\n        size_mb = round(p.stat().st_size / 1e6, 3)\n        self.manifest[p.name] = {\n            "stage": stage, "params": params, "meta": meta or {},\n            "git_commit": _git_hash(),\n            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),\n            "size_mb": size_mb,\n        }\n        self.manifest_path.write_text(\n            json.dumps(self.manifest, indent=2, default=str))\n        if self.verbose:\n            print("[ckpt] saved " + p.name + "  (" + str(size_mb) + " MB)")\n        return p\n\n    def load(self, stage, params=None):\n        p = self._path(stage, params)\n        if not p.exists():\n            raise FileNotFoundError("No checkpoint: " + str(p))\n        if self.verbose:\n            print("[ckpt] loaded " + p.name)\n        return joblib.load(p)\n\n    def cache(self, stage, fn, params=None, meta=None, force=False):\n        if self.exists(stage, params) and not force:\n            return self.load(stage, params)\n        if self.verbose:\n            print("[ckpt] computing " + stage + " ...")\n        out = fn()\n        self.save(stage, out, params=params, meta=meta)\n        return out\n\n    def summary(self):\n        import pandas as pd\n        if not self.manifest:\n            return pd.DataFrame()\n        return pd.DataFrame(self.manifest).T[\n            ["stage", "saved_at", "size_mb", "git_commit"]]\n\n    @classmethod\n    def latest_run(cls, root="/content/phishing-detection/checkpoints", **kw):\n        runs = sorted(p.name for p in Path(root).glob("run_*") if p.is_dir())\n        if not runs:\n            raise FileNotFoundError("No previous runs.")\n        return cls(root=root, run_id=runs[-1], **kw)\n'

target = REPO_DIR / "src" / "utils" / "checkpoint.py"
backup = DRIVE_DIR / "_repo_backup" / "src" / "utils" / "checkpoint.py"

if backup.exists():
    target.write_text(backup.read_text())
    print("restored checkpoint.py from Drive backup")
else:
    target.write_text(CKPT_SRC)
    print("wrote checkpoint.py from embedded source")

import importlib, src.utils.checkpoint as _ck
importlib.reload(_ck)
from src.utils.checkpoint import Checkpoint
print("Checkpoint import OK ->", target)

wrote checkpoint.py from embedded source
Checkpoint import OK -> /content/phishing-detection/src/utils/checkpoint.py


## Cell 2 — Acquisition module

Writes `src/data/acquire.py`. One generic Mendeley downloader serves all three
Mendeley datasets; UCI has its own path. Each function retries with backoff and
degrades to printed manual instructions rather than crashing the notebook.

In [3]:
%%writefile /content/phishing-detection/src/data/acquire.py
"""Dataset acquisition with integrity verification and provenance logging."""
import io, json, time, zipfile, hashlib, urllib.request, urllib.error
from pathlib import Path
import pandas as pd

REPO = Path("/content/phishing-detection")
RAW  = REPO / "data" / "raw"
EXT  = REPO / "data" / "external"
UA   = {"User-Agent": "Mozilla/5.0 (academic research; phishing-detection-study)"}

REGISTRY = {
    "phiusiil":       {"kind": "uci",      "id": 967,
                       "rows": 235795, "label": "label",     "phish": 0},
    "uci2012":        {"kind": "uci",      "id": 327,
                       "rows": 11055,  "label": "Result",    "phish": -1},
    "uci_multiclass": {"kind": "uci",      "id": 379,
                       "rows": 1353,   "label": "Result",    "phish": -1},
    "hannousse":      {"kind": "mendeley", "id": "c2gw7fy2j4", "version": 3,
                       "rows": 11430,  "label": "status",    "phish": "phishing"},
    "vrbancic":       {"kind": "mendeley", "id": "72ptz43s9v", "version": 1,
                       "rows": 88647,  "label": "phishing",  "phish": 1},
    "tan":            {"kind": "mendeley", "id": "h3cgnj8hft", "version": 1,
                       "rows": 10000,  "label": "CLASS_LABEL", "phish": 1},
}


# ------------------------------------------------------------------ helpers
def _get(url, timeout=240, retries=3):
    last = None
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=UA)
            return urllib.request.urlopen(req, timeout=timeout).read()
        except Exception as e:
            last = e
            wait = 2 ** attempt
            print(f"    retry {attempt+1}/{retries} in {wait}s ({type(e).__name__})")
            time.sleep(wait)
    raise last


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def _largest_csv(folder):
    cands = [p for p in Path(folder).rglob("*.csv") if p.stat().st_size > 0]
    if not cands:
        return None
    return max(cands, key=lambda p: p.stat().st_size)


# --------------------------------------------------------------------- UCI
def _fetch_uci(key, spec, out):
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=spec["id"])
    return pd.concat([ds.data.features, ds.data.targets], axis=1)


# ---------------------------------------------------------------- Mendeley
def _mendeley_file_list(ds_id, version):
    endpoints = [
        f"https://data.mendeley.com/public-api/datasets/{ds_id}/files"
        f"?folder_id=root&version={version}",
        f"https://api.data.mendeley.com/datasets/{ds_id}/versions/{version}/files",
    ]
    for url in endpoints:
        try:
            data = json.loads(_get(url, timeout=90, retries=2))
            if isinstance(data, dict):
                data = data.get("results") or data.get("files") or []
            if data:
                return data
        except Exception as e:
            print(f"    endpoint failed: {type(e).__name__}")
    raise RuntimeError("no Mendeley endpoint responded")


def _fetch_mendeley(key, spec, out):
    files = _mendeley_file_list(spec["id"], spec["version"])
    got = 0
    for f in files:
        name = f.get("filename") or f.get("name") or ""
        link = ((f.get("content_details") or {}).get("download_url")
                or f.get("download_url") or f.get("url"))
        if not (name and link):
            continue
        blob = _get(link)
        (out / name).write_bytes(blob)
        print(f"    got {name} ({len(blob)/1e6:.1f} MB)")
        got += 1
        if name.lower().endswith(".zip"):
            try:
                with zipfile.ZipFile(out / name) as z:
                    z.extractall(out)
            except zipfile.BadZipFile:
                print(f"    (not a valid zip: {name})")
    if not got:
        raise RuntimeError("file list returned but no downloadable entries")

    target = _largest_csv(out)
    if target is None:
        raise RuntimeError("no CSV found after extraction")
    return pd.read_csv(target, low_memory=False)


# ------------------------------------------------------------------ public
MANUAL = {
    "hannousse": "https://data.mendeley.com/datasets/c2gw7fy2j4/3",
    "vrbancic":  "https://data.mendeley.com/datasets/72ptz43s9v/1",
    "tan":       "https://data.mendeley.com/datasets/h3cgnj8hft/1",
}


def acquire(key, force=False):
    """Download one dataset. Returns a DataFrame, or None on failure."""
    spec = REGISTRY[key]
    out = RAW / key
    out.mkdir(parents=True, exist_ok=True)
    csv = out / f"{key}.csv"

    if csv.exists() and not force:
        df = pd.read_csv(csv, low_memory=False)
        print(f"[{key}] cached  {df.shape}")
        return df

    print(f"[{key}] downloading ...")
    try:
        fetch = _fetch_uci if spec["kind"] == "uci" else _fetch_mendeley
        df = fetch(key, spec, out)
    except Exception as e:
        print(f"[{key}] FAILED: {type(e).__name__}: {e}")
        if key in MANUAL:
            print(f"    Manual fallback:")
            print(f"      1. Open {MANUAL[key]}")
            print(f"      2. Download the extracted-features CSV")
            print(f"      3. Save it as: {csv}")
            print(f"      4. Re-run this cell")
        return None

    df.to_csv(csv, index=False)
    print(f"[{key}] saved {df.shape} -> {csv.name}")
    return df


def acquire_tranco(force=False):
    """Benign reference list. Record the permanent list ID for citation."""
    out = EXT / "tranco"
    out.mkdir(parents=True, exist_ok=True)
    csv = out / "tranco_top1m.csv"
    if csv.exists() and not force:
        df = pd.read_csv(csv, names=["rank", "domain"])
        print(f"[tranco] cached  {df.shape}")
        return df

    url = "https://tranco-list.eu/top-1m.csv.zip"
    try:
        with zipfile.ZipFile(io.BytesIO(_get(url))) as z:
            df = pd.read_csv(z.open(z.namelist()[0]), names=["rank", "domain"])
    except Exception as e:
        print(f"[tranco] FAILED: {e}")
        print("    Manual: download top-1m.csv.zip from https://tranco-list.eu/")
        print(f"    and save the CSV as {csv}")
        return None

    df.to_csv(csv, index=False, header=False)
    (out / "PROVENANCE.txt").write_text(
        f"Source: {url}\nDownloaded: {pd.Timestamp.now():%Y-%m-%d %H:%M}\n"
        "ACTION REQUIRED: visit https://tranco-list.eu/ and record the permanent\n"
        "list ID for this date. Cite the ID, not the top-1m.csv URL.\n")
    print(f"[tranco] saved {df.shape}")
    return df


def verify(key, df):
    """Check row count against the published figure. Returns a report dict."""
    spec = REGISTRY[key]
    csv = RAW / key / f"{key}.csv"
    if df is None:
        return {"dataset": key, "status": "MISSING", "rows": None,
                "expected": spec["rows"], "cols": None, "sha256": None}
    ok = len(df) == spec["rows"]
    return {
        "dataset": key,
        "status": "ok" if ok else "ROW COUNT MISMATCH",
        "rows": len(df),
        "expected": spec["rows"],
        "cols": df.shape[1],
        "sha256": sha256(csv)[:16] if csv.exists() else None,
    }


def write_provenance(reports):
    p = RAW / "PROVENANCE.json"
    payload = {
        "generated": pd.Timestamp.now().isoformat(),
        "datasets": {r["dataset"]: r for r in reports},
        "registry": {k: {kk: vv for kk, vv in v.items()} for k, v in REGISTRY.items()},
    }
    p.write_text(json.dumps(payload, indent=2, default=str))
    print(f"provenance -> {p}")
    return p

Writing /content/phishing-detection/src/data/acquire.py


## Cell 3 — Run the downloads

In [4]:
import importlib
import src.data.acquire as aq
importlib.reload(aq)

KEYS = ["phiusiil", "uci2012", "uci_multiclass", "hannousse", "vrbancic", "tan"]

data = {}
for k in KEYS:
    data[k] = aq.acquire(k)

tranco = aq.acquire_tranco()

[phiusiil] cached  (235795, 55)
[uci2012] cached  (11055, 31)
[uci_multiclass] cached  (1353, 10)
[hannousse] cached  (11430, 89)
[vrbancic] cached  (88647, 112)
[tan] downloading ...
    got Phishing_Legitimate_full.arff (1.4 MB)
[tan] FAILED: RuntimeError: no CSV found after extraction
    Manual fallback:
      1. Open https://data.mendeley.com/datasets/h3cgnj8hft/1
      2. Download the extracted-features CSV
      3. Save it as: /content/phishing-detection/data/raw/tan/tan.csv
      4. Re-run this cell
[tranco] cached  (1000000, 2)


## Cell 4 — Integrity report

Row counts are checked against the published figures. A mismatch means a
truncated or wrong file — not a finding. SHA-256 prefixes let you prove months
from now that the file you analysed is the file you downloaded.

In [5]:
import pandas as pd

reports = [aq.verify(k, data[k]) for k in KEYS]
report_df = pd.DataFrame(reports)
print(report_df.to_string(index=False))

aq.write_provenance(reports)

failed = [r["dataset"] for r in reports if r["status"] != "ok"]
if failed:
    print("\nNeeds attention:", ", ".join(failed))
else:
    print("\nAll datasets verified.")

       dataset  status     rows  expected  cols           sha256
      phiusiil      ok 235795.0    235795  55.0 cb4e29fc9a16e20c
       uci2012      ok  11055.0     11055  31.0 a13fc4835e437bb3
uci_multiclass      ok   1353.0      1353  10.0 54012e6218bed248
     hannousse      ok  11430.0     11430  89.0 147210c292a9b818
      vrbancic      ok  88647.0     88647 112.0 9ff7a20f2efb5e87
           tan MISSING      NaN     10000   NaN             None
provenance -> /content/phishing-detection/data/raw/PROVENANCE.json

Needs attention: tan


## Cell 5 — Schema inspection

Confirm the label column is where the registry says it is *before* you write any
modelling code against it.

In [6]:
for k in KEYS:
    df = data[k]
    if df is None:
        print(f"--- {k}: MISSING ---")
        continue
    spec = aq.REGISTRY[k]
    lab = spec["label"]
    match = [c for c in df.columns if c.lower() == lab.lower()]
    print(f"--- {k} ---")
    print(f"  shape        : {df.shape}")
    print(f"  label column : {match[0] if match else 'NOT FOUND -> ' + str(list(df.columns[-3:]))}")
    if match:
        vc = df[match[0]].value_counts()
        print(f"  distribution : {dict(vc)}")
    print(f"  dtypes       : {dict(df.dtypes.value_counts().astype(int))}")

--- phiusiil ---
  shape        : (235795, 55)
  label column : label
  distribution : {1: np.int64(134850), 0: np.int64(100945)}
  dtypes       : {dtype('int64'): np.int64(41), dtype('float64'): np.int64(10), dtype('O'): np.int64(4)}
--- uci2012 ---
  shape        : (11055, 31)
  label column : result
  distribution : {1: np.int64(6157), -1: np.int64(4898)}
  dtypes       : {dtype('int64'): np.int64(31)}
--- uci_multiclass ---
  shape        : (1353, 10)
  label column : Result
  distribution : {-1: np.int64(702), 1: np.int64(548), 0: np.int64(103)}
  dtypes       : {dtype('int64'): np.int64(10)}
--- hannousse ---
  shape        : (11430, 89)
  label column : status
  distribution : {'legitimate': np.int64(5715), 'phishing': np.int64(5715)}
  dtypes       : {dtype('int64'): np.int64(74), dtype('float64'): np.int64(13), dtype('O'): np.int64(2)}
--- vrbancic ---
  shape        : (88647, 112)
  label column : phishing
  distribution : {0: np.int64(58000), 1: np.int64(30647)}
  dtypes    

## Cell 6 — Harmonise labels to `1 = phishing`

The six sources use four different conventions. Normalise once, here, and never
think about it again. Phishing becomes the positive class so precision, recall
and PR-AUC all refer to the thing you are detecting.

In [7]:
def harmonise(key, df):
    """Return (X, y) with y = 1 for phishing, 0 for legitimate."""
    if df is None:
        return None, None
    spec = aq.REGISTRY[key]
    col = [c for c in df.columns if c.lower() == spec["label"].lower()]
    if not col:
        raise KeyError(f"{key}: label column '{spec['label']}' not found")
    col = col[0]
    ser = df[col]
    if isinstance(spec["phish"], str):
        y = (ser.astype(str).str.lower() == spec["phish"].lower()).astype(int)
    else:
        y = (ser == spec["phish"]).astype(int)
    X = df.drop(columns=[col])
    return X, y


harmonised = {}
rows = []
for k in KEYS:
    X, y = harmonise(k, data[k])
    if X is None:
        continue
    harmonised[k] = {"X": X, "y": y}
    rows.append({"dataset": k, "n": len(y), "features": X.shape[1],
                 "phishing_rate": round(float(y.mean()), 4)})

print(pd.DataFrame(rows).to_string(index=False))

       dataset      n  features  phishing_rate
      phiusiil 235795        54         0.4281
       uci2012  11055        30         0.4431
uci_multiclass   1353         9         0.5188
     hannousse  11430        88         0.5000
      vrbancic  88647       111         0.3457


**Sanity check on the rates above.** `hannousse` and `tan` should sit at
0.50 (both are balanced by construction). `phiusiil` should be ~0.428.
`uci2012` should be ~0.443. If any of these is inverted (e.g. 0.572 instead of
0.428) the label mapping is backwards for that source.

## Cell 7 — Checkpoint the harmonised bundle

In [8]:
from src.utils.checkpoint import Checkpoint

ck = Checkpoint(run_id="data_acquisition")

bundle = ck.cache(
    "harmonised_v2",
    lambda: harmonised,
    params={"convention": "1=phishing", "sources": sorted(harmonised)},
    meta={"note": "six sources, labels normalised, no feature engineering yet"},
)
print({k: v["X"].shape for k, v in bundle.items()})
ck.summary()

[ckpt] computing harmonised_v2 ...
[ckpt] saved harmonised_v2__8ab8683c29.joblib  (24.399 MB)
{'phiusiil': (235795, 54), 'uci2012': (11055, 30), 'uci_multiclass': (1353, 9), 'hannousse': (11430, 88), 'vrbancic': (88647, 111)}


,stage,saved_at,size_mb,git_commit
harmonised_v2__8ab8683c29.joblib,harmonised_v2,2026-07-24 03:22:05,24.399,nogit


## Cell 8 — Generate the Data section table

Writes `reports/tables/DATASETS.md` — paste straight into the manuscript and
edit the prose around it.

In [9]:
CITATIONS = {
    "phiusiil":       "Prasad & Chandra (2024), UCI ML Repository, doi:10.1016/j.cose.2023.103545",
    "uci2012":        "Mohammad & McCluskey (2012), UCI ML Repository, doi:10.24432/C51W2X",
    "uci_multiclass": "Abdelhamid (2015), UCI ML Repository",
    "hannousse":      "Hannousse & Yahiouche (2021), Mendeley Data, doi:10.17632/c2gw7fy2j4.3",
    "vrbancic":       "Vrbancic (2020), Mendeley Data, doi:10.17632/72ptz43s9v.1",
    "tan":            "Tan (2018), Mendeley Data, doi:10.17632/h3cgnj8hft.1",
}

lines = ["# Datasets", "",
         "| Dataset | N | Features | Phishing rate | Source |",
         "|---|---|---|---|---|"]
for k in KEYS:
    if k not in harmonised:
        continue
    X, y = harmonised[k]["X"], harmonised[k]["y"]
    lines.append(f"| `{k}` | {len(y):,} | {X.shape[1]} | "
                 f"{y.mean():.3f} | {CITATIONS[k]} |")

lines += ["", "All datasets are released under CC BY 4.0. Labels were harmonised so that",
          "y = 1 denotes a phishing instance across all sources.", "",
          "Benign reference list: Tranco (record permanent list ID before submission).", ""]

out = REPO_DIR / "reports" / "tables" / "DATASETS.md"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text("\n".join(lines))
print("\n".join(lines))
print("\nwritten ->", out)

# Datasets

| Dataset | N | Features | Phishing rate | Source |
|---|---|---|---|---|
| `phiusiil` | 235,795 | 54 | 0.428 | Prasad & Chandra (2024), UCI ML Repository, doi:10.1016/j.cose.2023.103545 |
| `uci2012` | 11,055 | 30 | 0.443 | Mohammad & McCluskey (2012), UCI ML Repository, doi:10.24432/C51W2X |
| `uci_multiclass` | 1,353 | 9 | 0.519 | Abdelhamid (2015), UCI ML Repository |
| `hannousse` | 11,430 | 88 | 0.500 | Hannousse & Yahiouche (2021), Mendeley Data, doi:10.17632/c2gw7fy2j4.3 |
| `vrbancic` | 88,647 | 111 | 0.346 | Vrbancic (2020), Mendeley Data, doi:10.17632/72ptz43s9v.1 |

All datasets are released under CC BY 4.0. Labels were harmonised so that
y = 1 denotes a phishing instance across all sources.

Benign reference list: Tranco (record permanent list ID before submission).


written -> /content/phishing-detection/reports/tables/DATASETS.md


## Cell 9 — Disk usage and sync

In [10]:
!du -sh {DRIVE_DIR}/data/raw/* 2>/dev/null
!echo "---"
!du -sh {DRIVE_DIR}/data {DRIVE_DIR}/checkpoints 2>/dev/null

import shutil, glob
for src in glob.glob(str(DRIVE_DIR / "notebooks" / "*.ipynb")):
    shutil.copy2(src, REPO_DIR / "notebooks")
for folder in ["src", "config"]:
    dst = DRIVE_DIR / "_repo_backup" / folder
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(REPO_DIR / folder, dst)
print("synced repo code to Drive")

7.0M	/content/drive/MyDrive/research/phishing-detection/data/raw/hannousse
54M	/content/drive/MyDrive/research/phishing-detection/data/raw/phiusiil
2.0K	/content/drive/MyDrive/research/phishing-detection/data/raw/PROVENANCE.json
1.4M	/content/drive/MyDrive/research/phishing-detection/data/raw/tan
775K	/content/drive/MyDrive/research/phishing-detection/data/raw/uci2012
36K	/content/drive/MyDrive/research/phishing-detection/data/raw/uci_multiclass
64M	/content/drive/MyDrive/research/phishing-detection/data/raw/vrbancic
---
147M	/content/drive/MyDrive/research/phishing-detection/data
51M	/content/drive/MyDrive/research/phishing-detection/checkpoints
synced repo code to Drive


In [11]:
import subprocess
def commit(msg):
    subprocess.run(["git", "-C", str(REPO_DIR), "add", "-A"])
    r = subprocess.run(["git", "-C", str(REPO_DIR), "commit", "-m", msg],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)

commit("feat: six-source acquisition with integrity checks and provenance")

fatal: not a git repository (or any of the parent directories): .git



---
### Before moving on

1. Open `data/external/tranco/PROVENANCE.txt` and record the permanent Tranco list ID.
2. Confirm every row in the Cell 4 report reads `ok`.
3. Check the phishing rates in Cell 6 against the expected values noted above.

Next: `03_features_and_splits.ipynb` — chronological splits, the cross-dataset
train/test harness, and the benign-source bias control.